# Lista 5 — Zadanie 3: Eksploracja modeli encoder-only (20 pkt)

In [ ]:
import sys

!{sys.executable} -m pip install -q torch transformers datasets scikit-learn pandas matplotlib

In [ ]:
import sys
from pathlib import Path

TASK5_DIR = Path("..").resolve()
if str(TASK5_DIR) not in sys.path:
    sys.path.insert(0, str(TASK5_DIR))

import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from common.data import load_polemo_test
from common.labels import map_text_to_class
from common.metrics import evaluate_predictions, print_evaluation

## Krok 1: Przygotowanie danych

In [ ]:
examples = load_polemo_test()
sentences = [ex["sentence"] for ex in examples]
y_true = [ex["class"] for ex in examples]

device = 0 if torch.cuda.is_available() else -1
print(f"Próbek: {len(sentences)} | Urządzenie: {'GPU' if device == 0 else 'CPU'}")

## Analiza długości tekstów

In [ ]:
lengths = [len(s.split()) for s in sentences]

print(f"Najkrótszy tekst (słowa): {min(lengths)}")
print(f"Najdłuższy tekst (słowa): {max(lengths)}")
print(f"Średnia długość: {sum(lengths) / len(lengths):.1f}")
print(f"Mediana: {pd.Series(lengths).median():.0f}")
print(f"90. percentyl: {pd.Series(lengths).quantile(0.9):.0f}")

plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=30, edgecolor="black", alpha=0.7)
plt.xlabel("Liczba słów")
plt.ylabel("Liczba recenzji")
plt.title("Rozkład długości tekstów w zbiorze testowym")
plt.axvline(32, color="orange", linestyle=":", alpha=0.8, label="max_length=32")
plt.axvline(64, color="purple", linestyle=":", alpha=0.8, label="max_length=64")
plt.axvline(128, color="red", linestyle="--", label="max_length=128")
plt.axvline(512, color="green", linestyle="--", label="max_length=512")
plt.legend()
plt.tight_layout()
plt.show()

## Krok 2: Funkcja pomocnicza do eksperymentów

In [ ]:
BASE_MODEL = "Voicelab/herbert-base-cased-sentiment"
METRIC_COLS = ["accuracy", "f1_macro", "f1_weighted"]


def _device_str():
    return "cuda" if device == 0 else "cpu"


def _load_encoder(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.to(_device_str()).eval()
    return tokenizer, model


def _predict_labels(model, tokenizer, texts, max_length, temperature):
    inputs = tokenizer(
        texts,
        truncation=True,
        max_length=max_length,
        padding=True,
        return_tensors="pt",
    ).to(_device_str())

    with torch.no_grad():
        logits = model(**inputs).logits
        if temperature == 0:
            pred_ids = logits.argmax(dim=-1)
        else:
            pred_ids = (logits / temperature).softmax(dim=-1).argmax(dim=-1)

    labels = [model.config.id2label[i] for i in pred_ids.cpu().tolist()]
    return [map_text_to_class(label) or "neutral" for label in labels]


def run_encoder_experiment(
    model_name,
    max_length=512,
    batch_size=16,
    temperature=1.0,
    model_cache=None,
):
    if model_cache is None:
        tokenizer, model = _load_encoder(model_name)
        model_cache = (tokenizer, model)
    else:
        tokenizer, model = model_cache

    y_pred = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        y_pred.extend(_predict_labels(model, tokenizer, batch, max_length, temperature))

    metrics = evaluate_predictions(y_true, y_pred)
    return metrics, y_pred, model_cache


def results_to_row(eksperyment, wariant, metrics):
    return {
        "eksperyment": eksperyment,
        "wariant": str(wariant),
        **{col: metrics[col] for col in METRIC_COLS},
    }


def style_results_table(df):
    return (
        df.style.format({col: "{:.4f}" for col in METRIC_COLS})
        .highlight_max(
            subset=METRIC_COLS,
            axis=0,
            props="font-weight: bold; background-color: #1565c0; color: #ffffff",
        )
    )


def show_results(df, title=None, plot=False):
    if title:
        print(title)
    display(style_results_table(df))
    if plot:
        plot_experiment_metrics(df, title)


def plot_experiment_metrics(df, title):
    labels = ["Accuracy", "F1 macro", "F1 weighted"]
    x = range(len(df))
    width = 0.25
    fig, ax = plt.subplots(figsize=(max(6, len(df) * 1.8), 4))

    for i, col in enumerate(METRIC_COLS):
        ax.bar([pos + i * width for pos in x], df[col], width=width, label=labels[i])

    ax.set_xticks([pos + width for pos in x])
    ax.set_xticklabels(df["wariant"], rotation=15, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Wartość metryki")
    ax.set_title(title)
    ax.legend(loc="lower right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

## Eksperyment A: Porównanie modeli

| Model | Opis |
|-------|------|
| `Voicelab/herbert-base-cased-sentiment` | HerBERT fine-tuned na sentyment ogólny (recenzje) |
| `bardsai/finance-sentiment-pl-base` | HerBERT fine-tuned na sentyment finansowy (inna domena) |

In [ ]:
MODELS_TO_COMPARE = [
    "Voicelab/herbert-base-cased-sentiment",
    "bardsai/finance-sentiment-pl-base",
]

model_results = []
for model_name in MODELS_TO_COMPARE:
    print(f"\n>>> {model_name}")
    try:
        metrics, _, _ = run_encoder_experiment(model_name)
        print_evaluation(metrics, title=model_name)
        model_results.append(results_to_row("model", model_name.split("/")[-1], metrics))
    except Exception as e:
        print(f"Błąd: {e}")

show_results(pd.DataFrame(model_results), "Eksperyment A — porównanie modeli", plot=True)

## Eksperyment B: Wpływ max_length

In [ ]:
MAX_LENGTHS = [32, 64, 128, 256, 512]

length_results = []
for max_len in MAX_LENGTHS:
    print(f"\n>>> max_length = {max_len}")
    metrics, _, _ = run_encoder_experiment(BASE_MODEL, max_length=max_len)
    print_evaluation(metrics, title=f"max_length={max_len}")
    length_results.append(results_to_row("max_length", max_len, metrics))

show_results(pd.DataFrame(length_results), "Eksperyment B — max_length")

## Eksperyment C: Wpływ temperatury

In [ ]:
TEMPERATURES = [0, 0.5, 1.0, 2.0]

temperature_results = []
temp_predictions = {}
model_cache = None

for temp in TEMPERATURES:
    print(f"\n>>> temperature = {temp}")
    metrics, y_pred, model_cache = run_encoder_experiment(
        BASE_MODEL, temperature=temp, model_cache=model_cache
    )
    temp_predictions[temp] = y_pred
    print_evaluation(metrics, title=f"temperature={temp}")
    temperature_results.append(results_to_row("temperature", temp, metrics))

show_results(pd.DataFrame(temperature_results), "Eksperyment C — temperatura")

ref_preds = temp_predictions[1.0]
print("\nZmiany predykcji względem T=1.0:")
for temp in TEMPERATURES:
    changed = sum(p != r for p, r in zip(temp_predictions[temp], ref_preds))
    print(f"  T={temp}: {changed} ({100 * changed / len(ref_preds):.1f}%)")

## Krok 3: Tabela porównawcza

In [ ]:
comparison_df = pd.DataFrame(model_results + length_results + temperature_results)
show_results(comparison_df, "Wszystkie eksperymenty — zestawienie")
comparison_df